# Fine-Tune LLaMA 3 8B on the Bible with SageMaker JumpStart

This notebook:
1. Downloads the King James Bible from Project Gutenberg
2. Prepares training data in JSONL format
3. Fine-tunes Meta LLaMA 3 8B via SageMaker JumpStart on `ml.g5.12xlarge`
4. Deploys both the vanilla and fine-tuned models on `ml.g5.xlarge`
5. Compares outputs side by side

## 1. Setup

In [3]:
import boto3
import sagemaker
import json
import os
import requests
import re
import textwrap
from sagemaker.jumpstart.model import JumpStartModel
from sagemaker.jumpstart.estimator import JumpStartEstimator

# --- Session setup (using your YOUR_PROFILE profile) ---
boto_session = boto3.Session(profile_name="YOUR_PROFILE", region_name="us-east-1")
sm_session = sagemaker.Session(boto_session=boto_session)
region = boto_session.region_name
bucket = sm_session.default_bucket()
prefix = "bible-llm-finetune"

# Find SageMaker execution role from IAM
iam = boto_session.client("iam")
roles = iam.list_roles(PathPrefix="/")["Roles"]
sm_roles = [r for r in roles if any(
    "sagemaker" in p.get("Principal", {}).get("Service", "")
    for s in [r.get("AssumeRolePolicyDocument", {}).get("Statement", [])]
    for p in (s if isinstance(s, list) else [s])
)]
if sm_roles:
    role = sm_roles[0]["Arn"]
    print(f"Found SageMaker role: {role}")
else:
    # Fallback: set your role ARN manually
    role = input("No SageMaker role found. Enter your SageMaker execution role ARN: ")

print(f"Region:  {region}")
print(f"Bucket:  {bucket}")
print(f"Prefix:  {prefix}")
print(f"Role:    {role}")

Found SageMaker role: arn:aws:iam::XXXXXXXXXXXX:role/service-role/YOUR-SAGEMAKER-ROLE
Region:  us-east-1
Bucket:  sagemaker-us-east-1-XXXXXXXXXXXX
Prefix:  bible-llm-finetune
Role:    arn:aws:iam::XXXXXXXXXXXX:role/service-role/YOUR-SAGEMAKER-ROLE


## 2. Download the King James Bible

In [21]:
# Download the King James Bible from Project Gutenberg (public domain)
bible_url = "https://www.gutenberg.org/cache/epub/10/pg10.txt"
response = requests.get(bible_url)
response.raise_for_status()
raw_text = response.text

# Strip the Project Gutenberg header/footer
start_marker = "*** START OF THE PROJECT GUTENBERG EBOOK"
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"
start_idx = raw_text.find(start_marker)
end_idx = raw_text.find(end_marker)
if start_idx != -1:
    raw_text = raw_text[raw_text.index("\n", start_idx) + 1:]
if end_idx != -1:
    raw_text = raw_text[:end_idx]

raw_text = raw_text.strip()
print(f"Bible text length: {len(raw_text):,} characters")
print(f"Preview: {raw_text[:500]}")

Bible text length: 4,433,037 characters
Preview: The Old Testament of the King James Version of the Bible
The First Book of Moses: Called Genesis
The Second Book of Moses: Called Exodus
The Third Book of Moses: Called Leviticus
The Fourth Book of Moses: Called Numbers
The Fifth Book of Moses: Called Deuteronomy
The Book of Joshua
The Book of Judges
The Book of Ruth
The First Book of Samuel
The Second Book of Samuel
The First Book of the Kings
The Second Book of the Kings
The First Book of the Chronicles
The Second Book of the Chr


## 3. Prepare Training Data (JSONL)

JumpStart expects JSONL with instruction-style entries. We split the Bible into
passages and create instruction/response pairs that teach the model Biblical style.

In [22]:
def extract_verses(text):
    """Split Bible text into book-chapter-verse chunks."""
    # Split into lines and group into passages of ~5-10 verses
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    passages = []
    current_passage = []
    
    for line in lines:
        current_passage.append(line)
        # Group roughly every 5-8 lines as a passage
        if len(current_passage) >= 6:
            passages.append(" ".join(current_passage))
            current_passage = []
    
    if current_passage:
        passages.append(" ".join(current_passage))
    
    return passages

passages = extract_verses(raw_text)
print(f"Total passages: {len(passages)}")
print(f"\nSample passage:\n{passages[10]}")

Total passages: 12455

Sample passage:
The General Epistle of James The First Epistle General of Peter The Second General Epistle of Peter The First Epistle General of John The Second Epistle General of John The Third Epistle General of John


In [23]:
# Create instruction-tuning dataset in JumpStart LLaMA format
# Expected keys: instruction, context, response
import random
random.seed(42)

prompt_templates = [
    "Continue this passage in the style of the King James Bible.",
    "Write in the style of the Holy Scriptures about the following passage.",
    "Speak as the scriptures would on this matter.",
    "Complete this Biblical text.",
    "In the manner of the King James Bible, teach about this passage.",
    "Recite the scripture style continuation.",
    "What does the Bible say? Continue from this passage.",
]

training_data = []

for passage in passages:
    words = passage.split()
    if len(words) < 20:
        continue

    split_point = len(words) // 3
    prompt_context = " ".join(words[:split_point])
    completion = " ".join(words[split_point:])

    entry = {
        "instruction": random.choice(prompt_templates),
        "context": prompt_context,
        "response": completion,
    }
    training_data.append(entry)

print(f"Training examples: {len(training_data)}")
print("\nSample entry:")
print(json.dumps(training_data[5], indent=2))
print("\nKeys in sample:", list(training_data[5].keys()))

Training examples: 12453

Sample entry:
{
  "instruction": "Write in the style of the Holy Scriptures about the following passage.",
  "context": "The Gospel According to Saint Mark The Gospel According to Saint Luke The Gospel",
  "response": "According to Saint John The Acts of the Apostles The Epistle of Paul the Apostle to the Romans The First Epistle of Paul the Apostle to the Corinthians"
}

Keys in sample: ['instruction', 'context', 'response']


In [24]:
# Save to local file and upload to S3
os.makedirs("data", exist_ok=True)
train_file = "data/bible_train.jsonl"

with open(train_file, "w") as f:
    for entry in training_data:
        f.write(json.dumps(entry) + "\n")

print(f"Saved {len(training_data)} examples to {train_file}")
file_size_mb = os.path.getsize(train_file) / (1024 * 1024)
print(f"File size: {file_size_mb:.2f} MB")

Saved 12453 examples to data/bible_train.jsonl
File size: 5.30 MB


In [25]:
# Upload training data to S3
s3_train_uri = sm_session.upload_data(
    path=train_file,
    bucket=bucket,
    key_prefix=f"{prefix}/train"
)
print(f"Training data uploaded to: {s3_train_uri}")

Training data uploaded to: s3://sagemaker-us-east-1-XXXXXXXXXXXX/bible-llm-finetune/train/bible_train.jsonl


## 4. Fine-Tune LLaMA 3 8B with JumpStart

Using `ml.g5.12xlarge` (4x NVIDIA A10G GPUs, 24 GB each; 96 GB total VRAM, plus higher vCPU/RAM for data loading).

In [26]:
# JumpStart model ID for Meta LLaMA 3 8B
model_id = "meta-textgeneration-llama-3-8b"

estimator = JumpStartEstimator(
    model_id=model_id,
    role=role,
    instance_type="ml.g5.12xlarge",
    instance_count=1,
    sagemaker_session=sm_session,
    environment={"accept_eula": "true"},
)

# Override key hyperparameters
estimator.set_hyperparameters(
    instruction_tuned="True",
    epoch="3",
    max_input_length="1024",
    per_device_train_batch_size="2",
    learning_rate="0.0001",
    lora_r="8",
    lora_alpha="32",
    lora_dropout="0.05",
)

print("Estimator configured.")
print("Instance type: ml.g5.12xlarge")
print(f"Model: {model_id}")

Model 'meta-textgeneration-llama-3-8b' requires accepting end-user license agreement (EULA). See https://jumpstart-cache-prod-us-east-1.s3.us-east-1.amazonaws.com/fmhMetadata/eula/llama3Eula.txt for terms of use.
INFO:sagemaker.jumpstart:Model 'meta-textgeneration-llama-3-8b' requires accepting end-user license agreement (EULA). See https://jumpstart-cache-prod-us-east-1.s3.us-east-1.amazonaws.com/fmhMetadata/eula/llama3Eula.txt for terms of use.
Using model 'meta-textgeneration-llama-3-8b' with wildcard version identifier '*'. You can pin to version '3.0.0' for more stable results. Note that models may have different input/output signatures after a major version upgrade.


Estimator configured.
Instance type: ml.g5.12xlarge
Model: meta-textgeneration-llama-3-8b


In [27]:
# Launch the fine-tuning job (takes ~1-2 hours)
estimator.fit({"training": s3_train_uri})
print("Fine-tuning complete!")

INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
INFO:sagemaker:Creating training-job with name: meta-textgeneration-llama-3-8b-2026-04-04-01-57-09-940


2026-04-04 01:57:10 Starting - Starting the training job
2026-04-04 01:57:10 Pending - Training job waiting for capacity..................
2026-04-04 02:00:11 Downloading - Downloading input data.............................bash: cannot set terminal process group (-1): Inappropriate ioctl for device
bash: no job control in this shell
2026-04-04 02:04:57,901 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2026-04-04 02:04:57,940 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2026-04-04 02:04:57,948 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2026-04-04 02:04:57,950 sagemaker_pytorch_container.training INFO     Invoking user training script.

2026-04-04 02:04:54 Training - Training image download completed. Training in progress.2026-04-04 02:05:06,892 sagemaker-training-toolkit INFO     Installing dependencies from requirements.txt:
/opt/conda/bin/python3.10 -

## 5. Deploy Both Models for Comparison

- **Vanilla**: Original LLaMA 3 8B from JumpStart
- **Fine-tuned**: Our Bible-tuned version

Both on `ml.g5.xlarge`.

In [5]:
# Deploy the VANILLA (base) model
# Keep model_id here so this cell works even if training-setup cells were skipped after restart.
model_id = "meta-textgeneration-llama-3-8b"

vanilla_model = JumpStartModel(
    model_id=model_id,
    role=role,
    sagemaker_session=sm_session,
)

vanilla_predictor = vanilla_model.deploy(
    initial_instance_count=1,
    instance_type="ml.g5.xlarge",
    endpoint_name="llama3-8b-vanilla-bible-test",
    accept_eula=True,
)
print("Vanilla model deployed!")

Using model 'meta-textgeneration-llama-3-8b' with wildcard version identifier '*'. You can pin to version '3.0.0' for more stable results. Note that models may have different input/output signatures after a major version upgrade.
Overriding instance type to ml.g5.xlarge


--------------------!Vanilla model deployed!


In [8]:
# Option A: Re-attach the completed fine-tuning job (no retraining)
# Use JumpStartEstimator.attach so deployment uses the correct JumpStart inference image.
model_id = "meta-textgeneration-llama-3-8b"
training_job_name = "meta-textgeneration-llama-3-8b-2026-04-04-01-57-09-940"
attached_estimator = JumpStartEstimator.attach(
    training_job_name,
    model_id=model_id,
    sagemaker_session=sm_session,
)
print(f"Attached to completed training job: {training_job_name}")


2026-04-04 03:41:53 Starting - Starting the training job
2026-04-04 03:41:53 Pending - Preparing the instances for training
2026-04-04 03:41:53 Downloading - Downloading the training image
2026-04-04 03:41:53 Training - Training image download completed. Training in progress.
2026-04-04 03:41:53 Uploading - Uploading generated training model
2026-04-04 03:41:53 Completed - Training job completed
Attached to completed training job: meta-textgeneration-llama-3-8b-2026-04-04-01-57-09-940


In [9]:
# Deploy the FINE-TUNED model from the completed training job (no retraining)
# Give SageMaker extra time to download and start the fine-tuned model.
finetuned_predictor = attached_estimator.deploy(
    initial_instance_count=1,
    instance_type="ml.g5.xlarge",
    endpoint_name="llama3-8b-bible-finetuned-v2",
    model_data_download_timeout=1200,
    container_startup_health_check_timeout=1200,
    env={"accept_eula": "true"},
)
print("Fine-tuned model deployed!")

Overriding instance type to ml.g5.xlarge
Overriding instance type to ml.g5.xlarge


--------------------!Fine-tuned model deployed!


## 6. Compare Outputs

In [ ]:
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer

# Ensure both predictors use JSON serialization
vanilla_predictor.serializer = JSONSerializer()
vanilla_predictor.deserializer = JSONDeserializer()
finetuned_predictor.serializer = JSONSerializer()
finetuned_predictor.deserializer = JSONDeserializer()


def query_endpoint(predictor, prompt, max_tokens=256):
    """Send a prompt to a SageMaker endpoint and return the response."""
    payload = {
        "inputs": prompt,
        "parameters": {
            "max_new_tokens": max_tokens,
            "temperature": 0.7,
            "top_p": 0.9,
            "do_sample": True,
        }
    }
    response = predictor.predict(payload)
    # Response can be a list of dicts, a single dict, or a string
    if isinstance(response, list):
        return response[0].get("generated_text", str(response))
    if isinstance(response, dict):
        return response.get("generated_text", str(response))
    return str(response)


def compare(prompt):
    """Run the same prompt through both models and print side by side."""
    print("=" * 80)
    print(f"PROMPT: {prompt}")
    print("=" * 80)
    
    vanilla_out = query_endpoint(vanilla_predictor, prompt)
    finetuned_out = query_endpoint(finetuned_predictor, prompt)
    
    print("\n--- VANILLA LLaMA 3 8B ---")
    print(textwrap.fill(vanilla_out, width=80))
    
    print("\n--- BIBLE FINE-TUNED LLaMA 3 8B ---")
    print(textwrap.fill(finetuned_out, width=80))
    print()

In [13]:
# Test prompts
test_prompts = [
    "In the beginning, there was",
    "Speak in the style of the King James Bible about the importance of patience:",
    "And the Lord spoke unto his people, saying:",
    "Tell me about forgiveness in the style of scripture:",
    "What is the meaning of love?",
]

for prompt in test_prompts:
    compare(prompt)

PROMPT: In the beginning, there was

--- VANILLA LLaMA 3 8B ---


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:11                                                                                   │
│                                                                                                  │
│    8 ]                                                                                           │
│    9                                                                                             │
│   10 for prompt in test_prompts:                                                                 │
│ ❱ 11 │   compare(prompt)                                                                         │
│   12                                                                                             │
│                                                                                                  │
│ in compare:38                                                                                    │
│                                                                                                  │
│   35 │   finetuned_out = query_endpoint(finetuned_predictor, prompt)                             │
│   36 │                                                                                           │
│   37 │   print("\n--- VANILLA LLaMA 3 8B ---")                                                   │
│ ❱ 38 │   print(textwrap.fill(vanilla_out, width=80))                                             │
│   39 │                                                                                           │
│   40 │   print("\n--- BIBLE FINE-TUNED LLaMA 3 8B ---")                                          │
│   41 │   print(textwrap.fill(finetuned_out, width=80))                                           │
│                                                                                                  │
│ /opt/homebrew/Cellar/python@3.11/3.11.13/Frameworks/Python.framework/Versions/3.11/lib/python3.1 │
│ 1/textwrap.py:396 in fill                                                                        │
│                                                                                                  │
│   393 │   available keyword args to customize wrapping behaviour.                                │
│   394 │   """                                                                                    │
│   395 │   w = TextWrapper(width=width, **kwargs)                                                 │
│ ❱ 396 │   return w.fill(text)                                                                    │
│   397                                                                                            │
│   398 def shorten(text, width, **kwargs):                                                        │
│   399 │   """Collapse and truncate the given text to fit in the given width.                     │
│                                                                                                  │
│ /opt/homebrew/Cellar/python@3.11/3.11.13/Frameworks/Python.framework/Versions/3.11/lib/python3.1 │
│ 1/textwrap.py:368 in fill                                                                        │
│                                                                                                  │
│   365 │   │   more than 'self.width' columns, and return a new string                            │
│   366 │   │   containing the entire wrapped paragraph.                                           │
│   367 │   │   """                                                                                │
│ ❱ 368 │   │   return "\n".join(self.wrap(text))                                                  │
│   369                                                                                            │
│   370                                                                                            │
│   371 # -- Convenience interface --------------------------

## 7. Cleanup

**IMPORTANT**: Delete endpoints when done to stop incurring charges!

In [ ]:
# Uncomment to delete endpoints when done
# vanilla_predictor.delete_endpoint()
# finetuned_predictor.delete_endpoint()
# print("Both endpoints deleted.")